In [1]:
import os
import glob
import re
from nbconvert import HTMLExporter
from traitlets.config import Config
import nbformat

# Install beautifulsoup4 if not already installed
try:
    from bs4 import BeautifulSoup
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'beautifulsoup4'])
    from bs4 import BeautifulSoup

def update_ipynb_links_to_html(html_body):
    """
    Post-process HTML body to:
    1. Replace .ipynb links with .html equivalents, preserving fragments (#anchors).
    2. Replace YouTube thumbnail images with embedded iframes.
    3. Replace local directory/file links with GitHub repository links (EXCLUDING .html and .pdf links).
    4. Replace .md file links with GitHub blob URLs.
    5. Handle Git submodules - redirect to their actual repositories instead of main repo.
    
    Args:
        html_body (str): The HTML content as a string.
    
    Returns:
        str: Updated HTML body.
    """
    # Submodule mappings: local path prefix -> actual GitHub repo URL
    submodule_mappings = {
        'arduino-liquid-flow-snippets': 'https://github.com/kanishkasthana/arduino-liquid-flow-snippets/tree/master',
        'ivPID': 'https://github.com/kanishkasthana/ivPID/tree/master',
    }
    
    # First, handle .ipynb link replacement
    pattern = r'(href=")([^"]*\.)?(?P<before>[^"#\.]*)ipynb(?P<after>#?[^"]*)"'
    def replacer(match):
        return f'{match.group(1)}{match.group(2)}{match.group("before")}html{match.group("after")}"'
    
    updated_body = re.sub(pattern, replacer, html_body)
    
    # Parse with BeautifulSoup for more complex operations
    soup = BeautifulSoup(updated_body, 'html.parser')
    replacements = 0
    
    # Handle YouTube thumbnail replacement with iframes
    for a_tag in soup.find_all('a', href=re.compile(r'youtube\.com/watch\?v=([a-zA-Z0-9_-]+)')):
        img_tag = a_tag.find('img')
        if img_tag and 'src' in img_tag.attrs:
            img_src = img_tag['src']
            video_match = re.search(r'youtube\.com/watch\?v=([a-zA-Z0-9_-]+)', a_tag['href'])
            if video_match:
                video_id = video_match.group(1)
                if re.search(rf'img\.youtube\.com/vi/{re.escape(video_id)}/[a-z]+default\.jpg', img_src):
                    responsive_div = soup.new_tag('div', style="position: relative; padding-bottom: 56.25%; height: 0; overflow: hidden; max-width: 100%; margin: 20px 0;")
                    responsive_div['class'] = 'youtube-embed'
                    
                    iframe = soup.new_tag('iframe',
                                          width="560",
                                          height="315",
                                          src=f"https://www.youtube.com/embed/{video_id}",
                                          title="YouTube video player",
                                          frameborder="0",
                                          allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share",
                                          referrerpolicy="strict-origin-when-cross-origin",
                                          allowfullscreen=True)
                    
                    iframe['style'] = "position: absolute; top: 0; left: 0; width: 100%; height: 100%;"
                    responsive_div.append(iframe)
                    a_tag.replace_with(responsive_div)
                    replacements += 1
                    print(f"Replaced YouTube thumbnail ({video_id}) with embedded iframe")
    
    # Handle local directory/file links
    github_base_url = "https://github.com/kanishkasthana/ChronoSeq"
    github_tree_url = f"{github_base_url}/tree/master"
    github_blob_url = f"{github_base_url}/blob/master"
    local_link_replacements = 0
    
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        
        if (href.startswith(('http://', 'https://', '#', 'mailto:')) or 
            '.html' in href or
            '.pdf' in href or
            'youtube.com' in href or
            'github.com' in href):
            continue
        
        if (href.startswith('./') or 
            href.startswith('../') or 
            ('/' in href and not href.startswith('/')) or
            (not href.startswith('/') and ('.' in href or len(href.split('/')) > 1))):
            
            clean_path = href.lstrip('./')
            
            if clean_path.startswith('../'):
                clean_path = clean_path.replace('../', '')
            
            # Check if path is within a submodule
            github_url = None
            for submodule_path, submodule_repo_url in submodule_mappings.items():
                if clean_path.startswith(submodule_path + '/') or clean_path == submodule_path:
                    subpath = clean_path[len(submodule_path):].lstrip('/')
                    if subpath:
                        github_url = f"{submodule_repo_url}/{subpath}"
                    else:
                        github_url = submodule_repo_url
                    print(f"Redirected submodule link: {href} -> {github_url}")
                    break
            
            if not github_url:
                if clean_path.endswith('.md'):
                    github_url = f"{github_blob_url}/{clean_path}"
                    print(f"Redirected .md file: {href} -> {github_url}")
                elif '.' in os.path.basename(clean_path):
                    github_url = f"{github_blob_url}/{clean_path}"
                    print(f"Redirected file: {href} -> {github_url}")
                else:
                    github_url = f"{github_tree_url}/{clean_path}"
                    print(f"Redirected directory: {href} -> {github_url}")
            
            a_tag['href'] = github_url
            a_tag['target'] = '_blank'
            local_link_replacements += 1
    
    if replacements > 0:
        print(f"Total YouTube thumbnails replaced with iframes: {replacements}")
    
    if local_link_replacements > 0:
        print(f"Total local directory/file links redirected to GitHub: {local_link_replacements}")
    
    return str(soup)

def convert_ipynb_to_html_with_link_updates(directory_path):
    """
    Recursively convert all .ipynb files to HTML, excluding:
    - ivPID directory
    - test directory (NEW)
    
    Features:
    1. .ipynb links -> .html links (with anchors)
    2. YouTube thumbnails -> embedded iframes (560x315)
    3. Local directory/file links -> GitHub repository links (EXCLUDING .html and .pdf links)
    4. .md files -> GitHub blob URLs
    5. Submodule paths -> Actual submodule repository URLs
    Special handling: Rename "OPEN ME FIRST - README - Main Menu.ipynb" output to root index.html.
    
    Args:
        directory_path (str): Path to the root directory to search for .ipynb files.
    """
    # Find all .ipynb files recursively
    ipynb_files = sorted(glob.glob(os.path.join(directory_path, '**', '*.ipynb'), recursive=True))
    
    if not ipynb_files:
        print("No .ipynb files found in the directory.")
        return
    
    # Directories to skip (case-insensitive matching)
    skip_dirs = {'ivpid', 'test'}
    
    # Filter out files in excluded directories
    filtered_files = []
    for ipynb_path in ipynb_files:
        rel_path = os.path.relpath(ipynb_path, directory_path)
        path_parts = rel_path.split(os.sep)
        
        # Check if any part of the path matches excluded directories
        if any(part.lower() in skip_dirs for part in path_parts):
            print(f"Skipping (excluded directory): {ipynb_path}")
            continue
        
        filtered_files.append(ipynb_path)
    
    if not filtered_files:
        print("No .ipynb files to convert after filtering excluded directories.")
        return
    
    # Configure exporter
    c = Config()
    c.HTMLExporter.embed_images = False
    exporter = HTMLExporter(config=c)
    
    root_dir = os.path.abspath(directory_path)
    target_notebook_name = "OPEN ME FIRST - README - Main Menu.ipynb"
    
    for ipynb_path in filtered_files:
        try:
            with open(ipynb_path, 'r', encoding='utf-8') as f:
                notebook = nbformat.read(f, as_version=4)
            
            body, resources = exporter.from_notebook_node(notebook)
            body = update_ipynb_links_to_html(body)
            
            notebook_name = os.path.basename(ipynb_path)
            if notebook_name == target_notebook_name:
                html_filename = 'index.html'
                html_path = os.path.join(root_dir, html_filename)
                resource_dir_name = 'index_files'
                print(f"Special conversion: Renaming to root {html_path}")
            else:
                html_dir = os.path.dirname(ipynb_path)
                html_filename = os.path.splitext(notebook_name)[0] + '.html'
                html_path = os.path.join(html_dir, html_filename)
                resource_dir_name = f"{os.path.splitext(html_filename)[0]}_files"
            
            with open(html_path, 'w', encoding='utf-8') as f:
                f.write(body)
            
            if resources and 'files' in resources:
                if notebook_name == target_notebook_name:
                    resource_dir = os.path.join(root_dir, resource_dir_name)
                else:
                    resource_dir = os.path.join(os.path.dirname(html_path), resource_dir_name)
                os.makedirs(resource_dir, exist_ok=True)
                for file_path, content in resources.get('files', {}).items():
                    with open(os.path.join(resource_dir, file_path), 'wb') as res_file:
                        res_file.write(content)
                print(f"Extracted attachments to: {resource_dir}")
            
            print(f"Converted: {ipynb_path} -> {html_path}")
            
        except Exception as e:
            print(f"Error converting {ipynb_path}: {str(e)}")

# Execute conversion
directory = '.'
convert_ipynb_to_html_with_link_updates(directory)


Skipping (excluded directory): .\test\test_pbmc_conditions_comparison.ipynb
Skipping (excluded directory): .\test\test_protocol_for_single_cell_scale_up_chronoseqv6_dropseq_bead_modification.ipynb
Redirected file: CAD_Files/3%20Port%20Reservoir/Screenshot%202024-05-18%20at%206.41.43%E2%80%AFPM.png -> https://github.com/kanishkasthana/ChronoSeq/blob/master/CAD_Files/3%20Port%20Reservoir/Screenshot%202024-05-18%20at%206.41.43%E2%80%AFPM.png
Redirected file: CAD_Files/3%20Port%20Reservoir/3%20Port%20Cap%20XY%20Compensation.3mf -> https://github.com/kanishkasthana/ChronoSeq/blob/master/CAD_Files/3%20Port%20Reservoir/3%20Port%20Cap%20XY%20Compensation.3mf
Total local directory/file links redirected to GitHub: 2
Converted: .\CAD_Files\3 Port Reservoir\Manufacturing Notes.ipynb -> .\CAD_Files\3 Port Reservoir\Manufacturing Notes.html
Converted: .\ChronoSeq_Overview.ipynb -> .\ChronoSeq_Overview.html
Replaced YouTube thumbnail (qbhCONpUdx0) with embedded iframe
Replaced YouTube thumbnail (J2O1

C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Redirected file: CAD_Files/Valve%20Controller%20ESP32/Relay%20Controller%20ESP32.3mf -> https://github.com/kanishkasthana/ChronoSeq/blob/master/CAD_Files/Valve%20Controller%20ESP32/Relay%20Controller%20ESP32.3mf
Redirected directory: CAD_Files/Valve%20Controller%20ESP32 -> https://github.com/kanishkasthana/ChronoSeq/tree/master/CAD_Files/Valve%20Controller%20ESP32
Redirected directory: relay_Control_esp32/ -> https://github.com/kanishkasthana/ChronoSeq/tree/master/relay_Control_esp32/
Total local directory/file links redirected to GitHub: 3
Converted: .\instructions_for_assembling_valve_controller_ESP32.ipynb -> .\instructions_for_assembling_valve_controller_ESP32.html
Converted: .\instructions_for_assembling_vortex_relay_controller.ipynb -> .\instructions_for_assembling_vortex_relay_controller.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Redirected directory: CAD_Files/XY%20Robot -> https://github.com/kanishkasthana/ChronoSeq/tree/master/CAD_Files/XY%20Robot
Redirected directory: CAD_Files/Ice%20Box%20Lids%20for%20Heathrow%20Scientific%20HS23271A -> https://github.com/kanishkasthana/ChronoSeq/tree/master/CAD_Files/Ice%20Box%20Lids%20for%20Heathrow%20Scientific%20HS23271A
Redirected directory: droplet_collector/ -> https://github.com/kanishkasthana/ChronoSeq/tree/master/droplet_collector/
Total local directory/file links redirected to GitHub: 3
Converted: .\instructions_for_assembling_xy_robot_and_ice_boxes.ipynb -> .\instructions_for_assembling_xy_robot_and_ice_boxes.html
Redirected directory: CAD_Files/XYZ%20Robot -> https://github.com/kanishkasthana/ChronoSeq/tree/master/CAD_Files/XYZ%20Robot
Redirected directory: CAD_Files/XYZ%20Robot -> https://github.com/kanishkasthana/ChronoSeq/tree/master/CAD_Files/XYZ%20Robot
Redirected directory: doesbot/ -> https://github.com/kanishkasthana/ChronoSeq/tree/master/doesbot/
Tota

Redirected directory: CAD_Files/Sorval%20ST8R%20CAD%20Models -> https://github.com/kanishkasthana/ChronoSeq/tree/master/CAD_Files/Sorval%20ST8R%20CAD%20Models
Redirected directory: CAD_Files/Sorval%20ST8R%20CAD%20Models -> https://github.com/kanishkasthana/ChronoSeq/tree/master/CAD_Files/Sorval%20ST8R%20CAD%20Models
Redirected file: Seq_Only_Sample_Info_Template_July2023.xlsx -> https://github.com/kanishkasthana/ChronoSeq/blob/master/Seq_Only_Sample_Info_Template_July2023.xlsx
Redirected file: Example_Sequencing_Manifest.xlsx -> https://github.com/kanishkasthana/ChronoSeq/blob/master/Example_Sequencing_Manifest.xlsx
Total local directory/file links redirected to GitHub: 4
Converted: .\protocol_library_preparation_for_dropseq_chronoseq_beads_previous_version.ipynb -> .\protocol_library_preparation_for_dropseq_chronoseq_beads_previous_version.html
Converted: .\qPCR_validation_files\K562_Costimulation_2_Samples\qPCR_analysis_publication_quality_plots.ipynb -> .\qPCR_validation_files\K562_

In [2]:
import os

def generate_sitemap_html(directory_path, output_file='sitemap.html', debug=False):
    """
    Generate a beautiful GitHub Pages compatible sitemap.
    Includes <base target="_top"> to break out of iframe embedding.
    Top 3 cards always expanded. Remaining cards have individual collapsible toggles.
    Dropdowns auto-open during search, auto-close when search is cleared.
    """
    root_dir = os.path.abspath(directory_path)
    
    ignored_basenames = {
        'chronoseq_overview', 'convertohtml', 'index', 
        'removelinksfromnbconverthtml', 'sitemap'
    }
    
    deprecated_hardware_files = {
        'instructions_for_assembling_valve_controller',
        'instructions_for_assembling_valve_controller_esp32'
    }
    
    deprecated_protocol_files = {
        'protocol_and_software_for_running_chronoseq_device-originaldevice',
        'protocol_for_chronoseqv3_and_dropseq_bead_modification_100um_or_10um',
        'protocol_for_chronoseqv3_and_dropseq_bead_modification_1mm',
        'protocol_for_chronoseqv4_bead_modification_for_bulk_time_series',
        'protocol_for_exo1_predigestion_for_chronoseqv2_and_dropseq_beads',
        'protocol_for_preparing_cells',
        'protocol_for_preparing_cells_experimentalusingbsa',
        'protocol_for_qpcr_checking_of_k562_costimulation',
        'protocol_for_single_cell_scale_up_chronoseqv4_dropseq_bead_modification',
        'protocol_library_preparation_for_dropseq_chronoseq_beads_previous_version'
    }
    
    main_hardware_links = {
        'assembly': 'instructions_for_device_assembly_and_setup.html',
        'operation': 'protocol_and_software_for_running_chronoseq_device.html'
    }
    
    chronoseq_repo = 'https://github.com/kanishkasthana/ChronoSeq'
    preprint_link = 'https://doi.org/10.1101/2025.11.15.688181'
    youtube_video = 'https://www.youtube.com/watch?v=8NZF_kFQE5k'
    device_gif = 'img/Simplified%20Figure.gif'
    discord_invite = 'https://discord.gg/4YcQKkynUY'
    github_discussions = 'https://github.com/kanishkasthana/ChronoSeq/discussions'
    kanishk_linktree = 'https://linktr.ee/kanishkasthana'
    
    # Collect files
    html_files = {}
    hardware_files = []
    deprecated_hardware_files_found = []
    protocol_files = []
    deprecated_protocol_files_found = []
    
    for root, dirs, files in os.walk(root_dir):
        rel_root = os.path.relpath(root, root_dir).replace(os.sep, '/')
        if 'ivPID' in rel_root.split('/'):
            dirs[:] = []
            continue
        for file in files:
            if file.endswith('.html'):
                basename_stripped = os.path.splitext(file)[0].lower()
                full_rel_path = os.path.join(rel_root, file).replace(os.sep, '/')
                if basename_stripped in ignored_basenames:
                    continue
                basename = file.lower()
                if basename.startswith('instructions'):
                    if basename_stripped in deprecated_hardware_files:
                        deprecated_hardware_files_found.append(full_rel_path)
                    else:
                        hardware_files.append(full_rel_path)
                elif basename.startswith('protocol'):
                    if basename_stripped in deprecated_protocol_files:
                        deprecated_protocol_files_found.append(full_rel_path)
                    else:
                        protocol_files.append(full_rel_path)
                else:
                    file_dir = rel_root if rel_root != '.' else ''
                    if file_dir not in html_files:
                        html_files[file_dir] = []
                    html_files[file_dir].append(full_rel_path)
    
    hardware_files = sorted(hardware_files)
    deprecated_hardware_files_found = sorted(deprecated_hardware_files_found)
    protocol_files = sorted(protocol_files)
    deprecated_protocol_files_found = sorted(deprecated_protocol_files_found)
    for dir_key in sorted(html_files.keys()):
        html_files[dir_key] = sorted(html_files[dir_key])
    
    total_files = (len(hardware_files) + len(deprecated_hardware_files_found) + 
                   len(protocol_files) + len(deprecated_protocol_files_found) + 
                   sum(len(files_list) for files_list in html_files.values()))
    
    if total_files == 0:
        print("No .html files found.")
        return
    
    # Modern HTML with <base target="_top"> to break out of iframes
    html_content = [
        '<!DOCTYPE html>',
        '<html lang="en">',
        '<head>',
        '<meta charset="UTF-8">',
        '<meta name="viewport" content="width=device-width, initial-scale=1.0">',
        '<base target="_top">',
        '<title>ChronoSeq Links</title>',
        '<style>',
        '*{margin:0;padding:0;box-sizing:border-box;}',
        'body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Oxygen,Ubuntu,sans-serif;background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);min-height:100vh;padding:2rem 1rem;color:#1a202c;}',
        '.container{max-width:1200px;margin:0 auto;}',
        '.header{text-align:center;margin-bottom:2.5rem;}',
        '.header-title{display:flex;align-items:center;justify-content:center;gap:1.5rem;margin-bottom:1rem;}',
        'h1{font-size:3rem;font-weight:800;color:white;margin-bottom:0;text-shadow:0 4px 6px rgba(0,0,0,0.1);}',
        '.header-gif-small{width:100px;height:auto;border-radius:0.5rem;box-shadow:0 2px 8px rgba(0,0,0,0.2);cursor:pointer;transition:all 0.3s ease;flex-shrink:0;}',
        '.header-gif-small:hover{transform:scale(1.08);}',
        '.subtitle{font-size:1.125rem;color:rgba(255,255,255,0.9);margin-bottom:1.5rem;}',
        '.header-buttons{display:flex;gap:1rem;justify-content:center;align-items:center;flex-wrap:wrap;}',
        '.view-github,.read-preprint,.build-montage{display:inline-flex;align-items:center;gap:0.75rem;padding:1rem 2rem;border-radius:9999px;text-decoration:none;font-weight:700;font-size:1.1rem;transition:all 0.3s;position:relative;overflow:hidden;}',
        '.view-github{background:white;color:#667eea;box-shadow:0 4px 6px rgba(0,0,0,0.1);}',
        '.view-github:hover{transform:translateY(-2px);box-shadow:0 6px 12px rgba(0,0,0,0.15);}',
        '.read-preprint{background:linear-gradient(135deg,#ff6b6b,#ee5a6f,#c44569);color:white;box-shadow:0 6px 20px rgba(238,90,111,0.4);}',
        '.read-preprint::before{content:"";position:absolute;top:50%;left:50%;width:0;height:0;border-radius:50%;background:rgba(255,255,255,0.3);transform:translate(-50%,-50%);transition:width 0.6s,height 0.6s;}',
        '.read-preprint:hover::before{width:300px;height:300px;}',
        '.read-preprint:hover{transform:translateY(-4px) scale(1.05);box-shadow:0 10px 30px rgba(238,90,111,0.6);}',
        '@keyframes pulse{0%,100%{box-shadow:0 6px 20px rgba(238,90,111,0.4);}50%{box-shadow:0 6px 30px rgba(238,90,111,0.7);}}',
        '.read-preprint{animation:pulse 2s infinite;}',
        '.build-montage{background:linear-gradient(135deg,#ff0000,#cc0000);color:white;box-shadow:0 6px 20px rgba(255,0,0,0.4);}',
        '.build-montage::before{content:"";position:absolute;top:50%;left:50%;width:0;height:0;border-radius:50%;background:rgba(255,255,255,0.2);transform:translate(-50%,-50%);transition:width 0.6s,height 0.6s;}',
        '.build-montage:hover::before{width:300px;height:300px;}',
        '.build-montage:hover{transform:translateY(-4px) scale(1.05);box-shadow:0 10px 30px rgba(255,0,0,0.6);}',
        '@keyframes gifExpand{0%{width:80px;height:auto;transform:scale(0.3);}50%{width:400px;height:auto;transform:scale(0.7);}100%{width:90vw;height:90vh;max-width:1000px;max-height:800px;transform:scale(1);}}',
        '@keyframes gifFadeIn{0%{opacity:0;}30%{opacity:0.3;}100%{opacity:1;}}',
        '@keyframes overlayFadeIn{0%{opacity:0;}100%{opacity:1;}}',
        '.gif-overlay{position:fixed;top:0;left:0;width:100%;height:100%;background:rgba(0,0,0,0.6);display:none;align-items:center;justify-content:center;z-index:1000;opacity:0;cursor:pointer;}',
        '.gif-overlay.active{display:flex;animation:overlayFadeIn 0.3s ease-out forwards;}',
        '.gif-overlay.closing{animation:overlayFadeOut 0.3s ease-out forwards;}',
        '@keyframes overlayFadeOut{0%{opacity:1;}100%{opacity:0;}}',
        '.gif-overlay img{width:90vw;height:90vh;max-width:1000px;max-height:800px;object-fit:contain;border-radius:1rem;box-shadow:0 10px 40px rgba(0,0,0,0.4);pointer-events:none;opacity:0;}',
        '.gif-overlay.active img{animation:gifExpand 1.2s cubic-bezier(0.25,0.46,0.45,0.94) forwards,gifFadeIn 1.2s ease-out forwards;}',
        '.gif-overlay.closing img{animation:none;}',
        '.searchbar{text-align:center;margin:2rem 0;}',
        '.search-input{width:100%;max-width:600px;padding:1rem 1.5rem;border:none;border-radius:9999px;font-size:1rem;box-shadow:0 4px 6px rgba(0,0,0,0.1);outline:none;background:white;}',
        '.search-input:focus{box-shadow:0 6px 12px rgba(0,0,0,0.15);}',
        '.sections{display:flex;flex-direction:column;gap:1.5rem;}',
        '.top-grid{display:grid;gap:1.5rem;grid-template-columns:repeat(auto-fit,minmax(320px,1fr));margin-bottom:1.5rem;}',
        '.card{background:white;border-radius:1rem;padding:1.5rem;box-shadow:0 4px 6px rgba(0,0,0,0.1);transition:all 0.3s;position:relative;overflow:hidden;}',
        '.card:hover{transform:translateY(-4px);box-shadow:0 12px 24px rgba(0,0,0,0.15);}',
        '.card::before{content:"";position:absolute;top:0;left:0;right:0;height:4px;background:linear-gradient(90deg,#667eea,#764ba2);}',
        '.card-header{display:flex;align-items:center;gap:0.75rem;margin-bottom:1rem;cursor:pointer;user-select:none;}',
        '.card-header.collapsible{position:relative;padding-right:2rem;}',
        '.card-header.collapsible::after{content:"▶";position:absolute;right:0;transition:transform 0.3s;font-size:1.2rem;color:#667eea;font-weight:bold;}',
        '.card-header.collapsible.open::after{transform:rotate(90deg);}',
        '.icon{font-size:2rem;}',
        '.card-title{font-size:1.25rem;font-weight:700;color:#1a202c;flex:1;}',
        '.card-subtitle{font-size:0.875rem;color:#718096;margin-bottom:1rem;}',
        '.card-content{max-height:none;overflow:visible;}',
        '.card-content.collapsed{max-height:0;overflow:hidden;margin-bottom:-1rem;}',
        '.card-content.collapsed .link-list,.card-content.collapsed .card-subtitle,.card-content.collapsed .dir-label,.card-content.collapsed .toggle,.card-content.collapsed .dep-content{display:none;}',
        '.link-list{list-style:none;}',
        '.link-item{margin:0.5rem 0;}',
        '.link-btn{display:block;padding:0.75rem 1rem;background:rgba(102,126,234,0.05);border-radius:0.5rem;color:#1a202c;text-decoration:none;font-weight:500;transition:all 0.2s;border-left:3px solid #667eea;}',
        '.link-btn:hover{background:rgba(102,126,234,0.15);transform:translateX(4px);}',
        '.link-btn.highlight{background:linear-gradient(90deg,#667eea,#764ba2);color:white;border-left-color:white;}',
        '.external-link{display:inline-flex;align-items:center;gap:0.5rem;}',
        '.external-link::after{content:"↗";font-size:0.875rem;}',
        '.discord-btn{background:rgba(88,101,242,0.1);border-left-color:#5865f2;color:#5865f2;}',
        '.discord-btn:hover{background:rgba(88,101,242,0.2);}',
        '.github-btn{background:rgba(0,0,0,0.05);border-left-color:#24292e;color:#24292e;}',
        '.github-btn:hover{background:rgba(0,0,0,0.1);}',
        '.toggle{cursor:pointer;display:inline-flex;align-items:center;gap:0.5rem;background:rgba(239,68,68,0.1);color:#dc2626;padding:0.5rem 1rem;border-radius:0.5rem;font-weight:600;margin:1rem 0;user-select:none;}',
        '.toggle:hover{background:rgba(239,68,68,0.15);}',
        '.toggle::before{content:"▶";transition:transform 0.3s;display:inline-block;}',
        '.toggle.open::before{transform:rotate(90deg);}',
        '.dep-content{max-height:0;overflow:hidden;transition:max-height 0.3s ease;margin-left:1rem;border-left:2px solid rgba(239,68,68,0.3);padding-left:1rem;}',
        '.dep-content.open{max-height:2000px;padding-top:1rem;}',
        '.dep-notice{background:rgba(239,68,68,0.1);padding:0.75rem 1rem;border-radius:0.5rem;margin-bottom:1rem;border-left:3px solid #dc2626;font-size:0.875rem;color:#dc2626;}',
        '.deprecated .link-btn{background:rgba(239,68,68,0.05);border-left-color:#dc2626;color:#dc2626;}',
        '.deprecated .link-btn:hover{background:rgba(239,68,68,0.15);}',
        '.dir-label{font-weight:600;margin:1rem 0 0.5rem;color:#667eea;}',
        '@media(max-width:768px){h1{font-size:2rem;}.top-grid{grid-template-columns:1fr;}.header-buttons{flex-direction:column;}.header-gif-small{width:80px;}.header-title{gap:1rem;}}',
        '</style>',
        '</head>',
        '<body>',
        '<div class="container">',
        '<div class="header">',
        '<div class="header-title">',
        '<h1>ChronoSeq Links</h1>',
        f'<img src="{device_gif}" alt="ChronoSeq Device" class="header-gif-small" id="deviceGif">',
        '</div>',
        f'<p class="subtitle">Explore the key resources and documentation for the ChronoSeq project</p>',
        '<div class="header-buttons">',
        f'<a href="{chronoseq_repo}" class="view-github">',
        '<svg width="24" height="24" fill="currentColor" viewBox="0 0 24 24"><path d="M12 0c-6.626 0-12 5.373-12 12 0 5.302 3.438 9.8 8.207 11.387.599.111.793-.261.793-.577v-2.234c-3.338.726-4.033-1.416-4.033-1.416-.546-1.387-1.333-1.756-1.333-1.756-1.089-.745.083-.729.083-.729 1.205.084 1.839 1.237 1.839 1.237 1.07 1.834 2.807 1.304 3.492.997.107-.775.418-1.305.762-1.604-2.665-.305-5.467-1.334-5.467-5.931 0-1.311.469-2.381 1.236-3.221-.124-.303-.535-1.524.117-3.176 0 0 1.008-.322 3.301 1.23.957-.266 1.983-.399 3.003-.404 1.02.005 2.047.138 3.006.404 2.291-1.552 3.297-1.23 3.297-1.23.653 1.653.242 2.874.118 3.176.77.84 1.235 1.911 1.235 3.221 0 4.609-2.807 5.624-5.479 5.921.43.372.823 1.102.823 2.222v3.293c0 .319.192.694.801.576 4.765-1.589 8.199-6.086 8.199-11.386 0-6.627-5.373-12-12-12z"/></svg>',
        'View on GitHub',
        '</a>',
        f'<a href="{preprint_link}" class="read-preprint">',
        '<svg width="24" height="24" fill="currentColor" viewBox="0 0 24 24" style="position:relative;z-index:1;"><path d="M14 2H6c-1.1 0-1.99.9-1.99 2L4 20c0 1.1.89 2 1.99 2H18c1.1 0 2-.9 2-2V8l-6-6zm2 16H8v-2h8v2zm0-4H8v-2h8v2zm-3-5V3.5L18.5 9H13z"/></svg>',
        '<span style="position:relative;z-index:1;">Read our preprint</span>',
        '</a>',
        f'<a href="{youtube_video}" class="build-montage" target="_blank" rel="noopener noreferrer">',
        '<svg width="24" height="24" fill="currentColor" viewBox="0 0 24 24" style="position:relative;z-index:1;"><path d="M23.498 6.186a3.016 3.016 0 0 0-2.122-2.136C19.505 3.545 12 3.545 12 3.545s-7.505 0-9.377.505A3.017 3.017 0 0 0 .502 6.186C0 8.07 0 12 0 12s0 3.93.502 5.814a3.016 3.016 0 0 0 2.122 2.136c1.871.505 9.376.505 9.376.505s7.505 0 9.377-.505a3.015 3.015 0 0 0 2.122-2.136C24 15.93 24 12 24 12s0-3.93-.502-5.814zM9.545 15.568V8.432L15.818 12l-6.273 3.568z"/></svg>',
        '<span style="position:relative;z-index:1;">Build montage</span>',
        '</a>',
        '</div>',
        '</div>',
        '<div id="gifOverlay" class="gif-overlay">',
        f'<img src="{device_gif}" alt="ChronoSeq Device Overview Expanded">',
        '</div>',
        '<div class="searchbar">',
        '<input type="text" id="searchInput" class="search-input" placeholder="🔍 Search protocols, hardware, or instructions...">',
        '</div>',
        '<div class="sections">',
        '<!-- Top 3 cards in grid, always expanded -->',
        '<div class="top-grid">'
    ]
    
    # Card 1: Main Hardware (ALWAYS EXPANDED)
    html_content.extend([
        '<div class="card">',
        '<div class="card-header">',
        '<span class="icon">🔧</span>',
        '<h2 class="card-title">Main Hardware Pages</h2>',
        '</div>',
        '<div class="card-content">',
        '<p class="card-subtitle">Essential hardware documentation</p>',
        '<ul class="link-list">',
        f'<li class="link-item"><a href="{main_hardware_links["assembly"]}" class="link-btn">Main Hardware Assembly Page</a></li>',
        f'<li class="link-item"><a href="{main_hardware_links["operation"]}" class="link-btn">Main Hardware Operation Page</a></li>',
        '</ul>',
        '</div>',
        '</div>'
    ])
    
    # Card 2: Data Processing (ALWAYS EXPANDED)
    html_content.extend([
        '<div class="card">',
        '<div class="card-header">',
        '<span class="icon">📊</span>',
        '<h2 class="card-title">Data Processing & Analysis</h2>',
        '</div>',
        '<div class="card-content">',
        '<p class="card-subtitle">Tools and repositories for data analysis</p>',
        '<ul class="link-list">',
        '<li class="link-item"><a href="https://github.com/kanishkasthana/ChronoSeq-Tools" class="link-btn external-link">ChronoSeq-Tools repo</a></li>',
        '<li class="link-item"><a href="https://github.com/kanishkasthana/ChronoSeq-QC" class="link-btn external-link">ChronoSeq-QC repo</a></li>',
        '<li class="link-item"><a href="https://github.com/anjambor/ChronoPack" class="link-btn external-link">ChronoPack repo</a></li>',
        '</ul>',
        '</div>',
        '</div>'
    ])
    
    # Card 3: Community (ALWAYS EXPANDED)
    html_content.extend([
        '<div class="card">',
        '<div class="card-header">',
        '<span class="icon">💬</span>',
        '<h2 class="card-title">Community</h2>',
        '</div>',
        '<div class="card-content">',
        '<p class="card-subtitle">Connect with ChronoSeq users and get support</p>',
        '<ul class="link-list">',
        f'<li class="link-item"><a href="{discord_invite}" class="link-btn discord-btn external-link">ChronoSeq Discord Server</a></li>',
        f'<li class="link-item"><a href="{github_discussions}" class="link-btn github-btn external-link">GitHub Discussions</a></li>',
        f'<li class="link-item"><a href="{kanishk_linktree}" class="link-btn external-link" target="_blank" rel="noopener noreferrer">Linktree</a></li>',
        '</ul>',
        '</div>',
        '</div>'
    ])
    
    html_content.append('</div>')
    
    # Hardware Assembly Card (COLLAPSIBLE)
    if hardware_files or deprecated_hardware_files_found:
        html_content.extend([
            '<div class="card">',
            '<div class="card-header collapsible" onclick="toggleCard(this)">',
            '<span class="icon">⚙️</span>',
            '<h2 class="card-title">Hardware Assembly</h2>',
            '</div>',
            '<div class="card-content collapsed">',
            '<p class="card-subtitle">Detailed assembly instructions for ChronoSeq hardware components</p>',
            '<ul class="link-list">'
        ])
        for rel_path in hardware_files:
            link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ').replace('_', ' ').title()
            html_content.append(f'<li class="link-item"><a href="{rel_path}" class="link-btn">{link_text}</a></li>')
        html_content.append('</ul>')
        
        if deprecated_hardware_files_found:
            html_content.extend([
                '<div class="toggle" onclick="toggleDep(event)">⚠️ Deprecated Instructions</div>',
                '<div class="dep-content">'
            ])
            html_content.extend([
                '<div class="dep-notice"><strong>Note:</strong> These instructions are deprecated and may not work with current versions.</div>',
                '<ul class="link-list deprecated">'
            ])
            for rel_path in deprecated_hardware_files_found:
                link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ').replace('_', ' ').title()
                html_content.append(f'<li class="link-item"><a href="{rel_path}" class="link-btn">{link_text}</a></li>')
            html_content.extend(['</ul>', '</div>'])
        
        html_content.extend(['</div>', '</div>'])
    
    # Protocols Card (COLLAPSIBLE)
    if protocol_files or deprecated_protocol_files_found:
        html_content.extend([
            '<div class="card">',
            '<div class="card-header collapsible" onclick="toggleCard(this)">',
            '<span class="icon">📋</span>',
            '<h2 class="card-title">Protocols</h2>',
            '</div>',
            '<div class="card-content collapsed">',
            '<p class="card-subtitle">Laboratory protocols and procedures</p>',
            '<ul class="link-list">'
        ])
        for rel_path in protocol_files:
            link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ').replace('_', ' ').title()
            html_content.append(f'<li class="link-item"><a href="{rel_path}" class="link-btn">{link_text}</a></li>')
        html_content.append('</ul>')
        
        if deprecated_protocol_files_found:
            html_content.extend([
                '<div class="toggle" onclick="toggleDep(event)">⚠️ Deprecated Protocols</div>',
                '<div class="dep-content">'
            ])
            html_content.extend([
                '<div class="dep-notice"><strong>Note:</strong> These protocols are deprecated and may not work with current versions.</div>',
                '<ul class="link-list deprecated">'
            ])
            for rel_path in deprecated_protocol_files_found:
                link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ').replace('_', ' ').title()
                html_content.append(f'<li class="link-item"><a href="{rel_path}" class="link-btn">{link_text}</a></li>')
            html_content.extend(['</ul>', '</div>'])
        
        html_content.extend(['</div>', '</div>'])
    
    # Other Resources Card (COLLAPSIBLE)
    if html_files:
        html_content.extend([
            '<div class="card">',
            '<div class="card-header collapsible" onclick="toggleCard(this)">',
            '<span class="icon">📁</span>',
            '<h2 class="card-title">Other Resources</h2>',
            '</div>',
            '<div class="card-content collapsed">',
            '<p class="card-subtitle">CAD files, notebooks, and validation files</p>'
        ])
        for dir_key in sorted(html_files.keys()):
            dir_display = dir_key.replace('/', ' > ').replace('_', ' ').title() if dir_key else 'Root Directory'
            html_content.append(f'<p class="dir-label">{dir_display}</p>')
            html_content.append('<ul class="link-list">')
            for rel_path in html_files[dir_key]:
                link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ').replace('_', ' ').title()
                html_content.append(f'<li class="link-item"><a href="{rel_path}" class="link-btn">{link_text}</a></li>')
            html_content.append('</ul>')
        html_content.extend(['</div>', '</div>'])
    
    # JavaScript
    html_content.extend([
        '</div>',
        '</div>',
        '<script>',
        'const deviceGif=document.getElementById("deviceGif");',
        'const gifOverlay=document.getElementById("gifOverlay");',
        'deviceGif.addEventListener("click",function(e){',
        'e.stopPropagation();',
        'gifOverlay.classList.remove("closing");',
        'gifOverlay.classList.add("active");',
        '});',
        'gifOverlay.addEventListener("click",function(){',
        'gifOverlay.classList.remove("active");',
        'gifOverlay.classList.add("closing");',
        'setTimeout(()=>{',
        'gifOverlay.classList.remove("closing");',
        '},300);',
        '});',
        'function toggleCard(header){',
        'const content=header.nextElementSibling;',
        'header.classList.toggle("open");',
        'content.classList.toggle("collapsed");',
        '}',
        'function toggleDep(e){',
        'e.stopPropagation();',
        'const btn=e.target.closest(".toggle");',
        'const content=btn.nextElementSibling;',
        'btn.classList.toggle("open");',
        'content.classList.toggle("open");',
        '}',
        'const searchInput=document.getElementById("searchInput");',
        'searchInput.addEventListener("input",function(){',
        'const q=this.value.toLowerCase().trim();',
        'const collapsibleCards=document.querySelectorAll(".card-header.collapsible");',
        'let hasSearchResults=false;',
        'document.querySelectorAll(".card").forEach(card=>{',
        'let m=false;',
        'card.querySelectorAll(".link-item").forEach(item=>{',
        'const t=item.textContent.toLowerCase();',
        'if(!q||t.includes(q)){item.style.display="list-item";item.querySelector(".link-btn").classList.toggle("highlight",q&&t.includes(q));m=true;hasSearchResults=true;}',
        'else{item.style.display="none";item.querySelector(".link-btn").classList.remove("highlight");}',
        '});',
        'card.style.display=m||!q?"block":"none";',
        '});',
        'collapsibleCards.forEach(header=>{',
        'const card=header.closest(".card");',
        'if(q){',
        'const hasMatches=Array.from(card.querySelectorAll(".link-item")).some(item=>item.style.display!=="none");',
        'if(hasMatches){',
        'header.classList.add("open");',
        'header.nextElementSibling.classList.remove("collapsed");',
        '}else{',
        'header.classList.remove("open");',
        'header.nextElementSibling.classList.add("collapsed");',
        '}',
        '}else{',
        'header.classList.remove("open");',
        'header.nextElementSibling.classList.add("collapsed");',
        '}',
        '});',
        '});',
        '</script>',
        '</body>',
        '</html>'
    ])
    
    output_path = os.path.join(root_dir, output_file)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(html_content))
    
    print(f"✨ Generated sitemap at: {output_path}")
    print(f"   - Removed GitHub Issues / Bug Reports link")
    print(f"   - Renamed Kanishk's Linktree to Linktree")

directory = '.'
generate_sitemap_html(directory, debug=False)


✨ Generated sitemap at: C:\Users\ChronoSeq\ChronoSeq\sitemap.html
   - Removed GitHub Issues / Bug Reports link
   - Renamed Kanishk's Linktree to Linktree
